In [1]:
# Lab Assignment 3: Topic Modeling using LDA
# Name: Mousab Mohammed Elhassan Adam Mohammed - Hashem Musaab Abdo Mohammed
# Student ID: SW01084034 - SW01084071

# 1. Import Required Libraries
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer, SnowballStemmer

from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel



## 2. Download NLTK Resources

Run this cell once. If the resources already exist, NLTK will skip downloading them.

In [2]:
# 2. Read the Dataset

import pandas as pd

df = pd.read_csv('news_dataset (1).csv')

# Use only the text column
text_df = df[['text']].copy()

# Remove null values BEFORE preprocessing
text_df = text_df.dropna()

print('Dataset shape:', df.shape)
print('Text-only dataset shape:', text_df.shape)

text_df.head()

Dataset shape: (11314, 5)
Text-only dataset shape: (11096, 1)


,text
0,I was wondering if anyone out there could enli...
1,I recently posted an article asking what kind ...
2,\nIt depends on your priorities. A lot of peo...
3,an excellent automatic can be found in the sub...
4,: Ford and his automobile. I need information...


## 3. Read the Dataset

Only the `text` column is used, as required by the lab assignment.

In [3]:
# 3. Text Pre-processing

english_stopwords = set(stopwords.words('english'))

custom_stopwords = {
    'would', 'could', 'also', 'one', 'two', 'said', 'say', 'says',
    'like', 'get', 'new', 'use', 'make', 'many', 'much', 'may',
    'even', 'still', 'going', 'go', 'see', 'know', 'think'
}

stop_words = english_stopwords.union(custom_stopwords)

lemmatizer = WordNetLemmatizer()
stemmer = SnowballStemmer('english')

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return 'a'
    elif treebank_tag.startswith('V'):
        return 'v'
    elif treebank_tag.startswith('N'):
        return 'n'
    elif treebank_tag.startswith('R'):
        return 'r'
    else:
        return 'n'

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words and len(token) > 2]
    
    tagged_tokens = pos_tag(tokens)
    lemmas = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged_tokens]
    stems = [stemmer.stem(word) for word in lemmas]
    
    return stems

text_df['tokens'] = text_df['text'].apply(preprocess_text)

# Remove empty documents after preprocessing
text_df = text_df[text_df['tokens'].map(len) > 0].copy()

processed_texts = text_df['tokens'].tolist()

text_df[['text', 'tokens']].head()

,text,tokens
0,I was wondering if anyone out there could enli...,"[wonder, anyon, enlighten, car, saw, day, door..."
1,I recently posted an article asking what kind ...,"[recent, post, articl, ask, kind, rate, singl,..."
2,\nIt depends on your priorities. A lot of peo...,"[depend, prioriti, lot, peopl, put, high, prio..."
3,an excellent automatic can be found in the sub...,"[excel, automat, find, subaru, legaci, switch,..."
4,: Ford and his automobile. I need information...,"[ford, automobil, need, inform, whether, ford,..."


## 4. Remove Null Values

In [4]:
text_df = text_df.dropna()

print("After removing nulls:", text_df.shape)

After removing nulls: (10992, 2)


## 5. Text Pre-processing

This version uses a slightly different approach from a basic solution:

- Remove URLs, emails, numbers, and punctuation
- Convert text to lowercase
- Tokenize text
- Remove stopwords
- Apply POS-aware lemmatization
- Apply Snowball stemming
- Remove very short tokens

In [5]:
processed_texts = text_df['tokens'].tolist()

## 6. Add Bigrams

Bigrams help keep common two-word phrases together, such as `stock_market` or `white_house`.

In [6]:
from gensim.models import Phrases
from gensim.models.phrases import Phraser

bigram = Phrases(processed_texts, min_count=2, threshold=5)
bigram_model = Phraser(bigram)

processed_texts = [bigram_model[text] for text in processed_texts]

processed_texts[:2]

[['wonder_anyon',
  'enlighten',
  'car',
  'saw',
  'day',
  'door_sport',
  'car',
  'look',
  'late_earli',
  'call',
  'bricklin',
  'door',
  'realli',
  'small',
  'addit',
  'front_bumper',
  'separ',
  'rest',
  'bodi',
  'anyon',
  'tellm',
  'model',
  'name',
  'engin',
  'spec',
  'year',
  'product',
  'car',
  'make',
  'histori',
  'whatev',
  'info',
  'funki',
  'look',
  'car',
  'pleas_mail'],
 ['recent_post',
  'articl',
  'ask',
  'kind',
  'rate',
  'singl',
  'male',
  'driver',
  'yrs_old',
  'pay',
  'perform',
  'car',
  'summari_repli',
  'receiv',
  'anymor',
  'close',
  'enough',
  'dodg_stealth',
  'twin',
  'turbo_model',
  'ticket',
  'accid',
  'hous',
  'take',
  'defens',
  'drive',
  'airbag',
  'ab',
  'secur',
  'alarm',
  'singl',
  'year',
  'decut',
  'state_farm',
  'insur',
  'includ',
  'addit',
  'umbrella',
  'polici',
  'car',
  'hous',
  'base',
  'polici',
  'standard',
  'polici',
  'requir',
  'defens',
  'drive',
  'cours',
  'less',

## 7. Create Dictionary and Corpus for Gensim

In [7]:
from gensim.corpora import Dictionary

dictionary = Dictionary(processed_texts)

dictionary.filter_extremes(no_below=2, no_above=0.5)

corpus = [dictionary.doc2bow(text) for text in processed_texts]

print("Documents:", len(corpus))
print("Vocabulary size:", len(dictionary))

Documents: 10992
Vocabulary size: 41461


## 8. Build the LDA Model using Gensim

The assignment asks us to find **4 topics**, so `num_topics=4`.

In [ ]:
from gensim.models import LdaModel

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=4,
    random_state=42,
    passes=10
)

for i, topic in lda_model.print_topics(num_words=10):
    print(f"Topic {i}: {topic}")

## 9. Evaluate the LDA Model using Coherence Score

In [ ]:
from gensim.models import CoherenceModel

coherence_model = CoherenceModel(
    model=lda_model,
    texts=processed_texts,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()

print("Coherence Score:", coherence_score)

## 10. Assign Each Document to Its Dominant Topic

In [ ]:
dominant_topics = []

for doc in corpus:
    topics = lda_model.get_document_topics(doc)
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    dominant_topics.append(dominant_topic)

results_df = text_df[['text']].copy()
results_df['dominant_topic'] = dominant_topics

results_df.head()

## 11. Summarize Topic Keywords

In [ ]:
topics = lda_model.print_topics(num_words=10)

topic_summary_df = pd.DataFrame(topics, columns=['Topic', 'Keywords'])

topic_summary_df

## 12. Topic Distribution

In [ ]:
import matplotlib.pyplot as plt

topic_counts = results_df['dominant_topic'].value_counts().sort_index()

plt.bar(topic_counts.index, topic_counts.values)
plt.xlabel("Topic")
plt.ylabel("Number of Documents")
plt.title("Topic Distribution")
plt.show()

## 13. Interpretation of Coherence Score

In [ ]:
student_name = 'Mousab Mohammed Elhassan Adam Mohammed'
student_id = 'SW01084034'

if coherence_score >= 0.60:
    quality_comment = 'This suggests that the topics are reasonably clear and interpretable.'
elif coherence_score >= 0.40:
    quality_comment = 'This suggests that the topics are moderately interpretable, but there is still room for improvement.'
else:
    quality_comment = 'This suggests that the topics are weak and may need better preprocessing or parameter tuning.'

interpretation = f'''
Name: {student_name}
Student ID: {student_id}

Interpretation:
The LDA model was used to discover 4 hidden topics from the news dataset using only the text column.
The coherence score obtained is {coherence_score:.4f}. Coherence score measures how meaningful and related
the top words in each topic are. {quality_comment}

Since this is an unsupervised task, results may vary depending on preprocessing and model parameters.
'''

print(interpretation)

## 14. Save the Topic Assignment Result

In [ ]:
results_df.to_csv('news_topic_assignment_results.csv', index=False)
topic_summary_df.to_csv('lda_topic_keywords.csv', index=False)

print("Files saved successfully!")